In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [3]:
import re
import unicodedata
import pandas as pd
import numpy as np

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import nltk
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer



TOP_K_CATEGORIES = 10
SAMPLE_PER_CLASS = 2000
RANDOM_STATE = 42


CONTEXT_COLUMNS = ["L1", "L2", "L3", "L4"]


C_VALUES = [0.1, 1, 10]

OUTER_FOLDS = 10
INNER_FOLDS = 3


USE_BERT = True
BERT_MODEL = "bert-base-uncased"
BERT_BATCH_SIZE = 32
BERT_MAX_LENGTH = 128


data preprocessing

In [4]:
data = pd.read_json("/content/drive/MyDrive/Colab Notebooks/SML/News_Category_Dataset_v3.json", lines=True)

data = data[["category", "headline", "short_description"]]

data = data.dropna()

data["category"] = data["category"].astype(str).str.strip()
data["headline"] = data["headline"].astype(str).str.strip()
data["short_description"] = data["short_description"].astype(str).str.strip()

data = data[
    (data["category"] != "") &
    (data["headline"] != "") &
    (data["short_description"] != "")
].copy()

top_10_categories = data["category"].value_counts().head(TOP_K_CATEGORIES).index.tolist()

data = data[data["category"].isin(top_10_categories)].copy()

sampled_data = []

for category in top_10_categories:
    category_data = data[data["category"] == category]
    category_sample = category_data.sample(n=SAMPLE_PER_CLASS, random_state=RANDOM_STATE)
    sampled_data.append(category_sample)

data = pd.concat(sampled_data)

data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

category_to_label = {}

for i in range(len(top_10_categories)):
    category_to_label[top_10_categories[i]] = i

labels = []

for category in data["category"]:
    labels.append(category_to_label[category])

data["label"] = labels
def get_first_tokens(text, number_of_tokens):
    tokens = str(text).split()
    return " ".join(tokens[:number_of_tokens])


data["L1"] = data["headline"].apply(lambda x: get_first_tokens(x, 5))

data["L2"] = data["headline"]

data["L3"] = data["headline"] + " " + data["short_description"].apply(
    lambda x: get_first_tokens(x, 15)
)

data["L4"] = data["headline"] + " " + data["short_description"]

data = data[
    [
        "category",
        "label",
        "headline",
        "short_description",
        "L1",
        "L2",
        "L3",
        "L4"
    ]
].copy()

data.to_csv("processed_news.csv", index=False)

label_rows = []

for category in top_10_categories:
    label_rows.append({
        "category": category,
        "label": category_to_label[category]
    })

label_mapping = pd.DataFrame(label_rows)

label_mapping.to_csv("label_mapping.csv", index=False)

print("processed_news.csv saved")
print("label_mapping.csv saved")
print("Data shape:", data.shape)
print(data["category"].value_counts())

processed_news.csv saved
label_mapping.csv saved
Data shape: (20000, 8)
category
PARENTING         2000
WELLNESS          2000
TRAVEL            2000
POLITICS          2000
FOOD & DRINK      2000
BUSINESS          2000
STYLE & BEAUTY    2000
HEALTHY LIVING    2000
ENTERTAINMENT     2000
QUEER VOICES      2000
Name: count, dtype: int64


Text preprocessing + TF-IDF + POS + BERT feature functions

In [5]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("N"):
        return wordnet.NOUN
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def clean_text_for_tfidf(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)

    tokens = word_tokenize(text)

    lemmatizer = WordNetLemmatizer()
    tagged_tokens = pos_tag(tokens)

    cleaned_tokens = []

    for word, tag in tagged_tokens:
        lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)
def extract_pos_features_one_text(text):
    if not isinstance(text, str):
        text = ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^\w\s]", " ", text)

    tokens = word_tokenize(text)

    if len(tokens) == 0:
        return [
            0, 0, 0, 0,
            0, 0, 0, 0,
            0, 0
        ]

    tagged_tokens = pos_tag(tokens)

    noun_count = 0
    verb_count = 0
    adjective_count = 0
    adverb_count = 0

    for word, tag in tagged_tokens:
        if tag.startswith("N"):
            noun_count += 1
        elif tag.startswith("V"):
            verb_count += 1
        elif tag.startswith("J"):
            adjective_count += 1
        elif tag.startswith("R"):
            adverb_count += 1

    token_count = len(tokens)

    word_lengths = []

    for token in tokens:
        word_lengths.append(len(token))

    average_word_length = np.mean(word_lengths)

    noun_ratio = noun_count / token_count
    verb_ratio = verb_count / token_count
    adjective_ratio = adjective_count / token_count
    adverb_ratio = adverb_count / token_count

    return [
        noun_count,
        verb_count,
        adjective_count,
        adverb_count,
        noun_ratio,
        verb_ratio,
        adjective_ratio,
        adverb_ratio,
        token_count,
        average_word_length
    ]


def make_pos_feature_matrix(text_list):
    feature_rows = []

    for text in text_list:
        features = extract_pos_features_one_text(text)
        feature_rows.append(features)

    return np.array(feature_rows, dtype=float)


def standardize_train_test(train_matrix, test_matrix):
    mean_values = train_matrix.mean(axis=0)
    std_values = train_matrix.std(axis=0)

    std_values[std_values == 0] = 1

    train_scaled = (train_matrix - mean_values) / std_values
    test_scaled = (test_matrix - mean_values) / std_values

    return train_scaled, test_scaled
def make_tfidf_matrices(X_train, X_test):
    cleaned_train = []

    for text in X_train:
        cleaned_train.append(clean_text_for_tfidf(text))

    cleaned_test = []

    for text in X_test:
        cleaned_test.append(clean_text_for_tfidf(text))

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=10000,
        stop_words="english"
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=15000
    )

    X_train_word = word_vectorizer.fit_transform(cleaned_train)
    X_test_word = word_vectorizer.transform(cleaned_test)

    X_train_char = char_vectorizer.fit_transform(cleaned_train)
    X_test_char = char_vectorizer.transform(cleaned_test)

    X_train_tfidf = hstack([X_train_word, X_train_char])
    X_test_tfidf = hstack([X_test_word, X_test_char])

    return X_train_tfidf, X_test_tfidf


def make_tfidf_pos_matrices(X_train, X_test):
    X_train_tfidf, X_test_tfidf = make_tfidf_matrices(X_train, X_test)

    X_train_pos = make_pos_feature_matrix(X_train)
    X_test_pos = make_pos_feature_matrix(X_test)

    X_train_pos, X_test_pos = standardize_train_test(X_train_pos, X_test_pos)

    X_train_pos = csr_matrix(X_train_pos)
    X_test_pos = csr_matrix(X_test_pos)

    X_train_combined = hstack([X_train_tfidf, X_train_pos])
    X_test_combined = hstack([X_test_tfidf, X_test_pos])

    return X_train_combined, X_test_combined
def make_bert_embeddings(text_list, model_name="bert-base-uncased", batch_size=32, max_length=128):
    import torch
    from transformers import AutoTokenizer, AutoModel


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("BERT device:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    model.eval()             

    all_embeddings = []
    total = len(text_list)

    for start_index in range(0, total, batch_size):
        batch_texts = text_list[start_index:start_index + batch_size]

        encoded = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        
        token_embeddings = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"]

        expanded_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        summed_embeddings = torch.sum(token_embeddings * expanded_mask, dim=1)
        summed_mask = torch.clamp(expanded_mask.sum(dim=1), min=1e-9)
        mean_embeddings = summed_embeddings / summed_mask

        
        all_embeddings.append(mean_embeddings.cpu().numpy())

        
        batch_index = start_index // batch_size
        if batch_index % 50 == 0:
            print(f"  BERT progress: {start_index}/{total}")

    all_embeddings = np.vstack(all_embeddings)
    return all_embeddings

CV + metrics

In [6]:
def make_stratified_folds(y, number_of_folds, random_state):
    y = np.array(y)

    rng = np.random.default_rng(random_state)

    folds = []

    for i in range(number_of_folds):
        folds.append([])

    unique_labels = np.unique(y)

    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)

        split_indices = np.array_split(label_indices, number_of_folds)

        for fold_number in range(number_of_folds):
            folds[fold_number].extend(split_indices[fold_number].tolist())

    final_folds = []

    for fold in folds:
        fold = np.array(fold)
        rng.shuffle(fold)
        final_folds.append(fold)

    return final_folds


def calculate_accuracy(y_true, y_pred):
    correct_count = 0

    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_count += 1

    return correct_count / len(y_true)


def calculate_macro_f1(y_true, y_pred):
    labels = np.unique(y_true)

    f1_scores = []

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        f1_scores.append(f1)

    return np.mean(f1_scores)


def calculate_weighted_f1(y_true, y_pred):
    labels = np.unique(y_true)

    total_count = len(y_true)
    weighted_sum = 0

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0
        support = 0

        for i in range(len(y_true)):
            if y_true[i] == label:
                support += 1

            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        weighted_sum += f1 * support

    return weighted_sum / total_count


def calculate_per_class_f1(y_true, y_pred):
    labels = np.unique(y_true)

    result = {}

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        result[int(label)] = f1

    return result

Define Model Training and Nested Cross-Validation Functions

In [7]:
def train_and_predict(X_train_text, y_train, X_test_text, representation, C_value, bert_train=None, bert_test=None):
    if representation == "tfidf":
        X_train_features, X_test_features = make_tfidf_matrices(
            X_train_text,
            X_test_text
        )

    elif representation == "tfidf_pos":
        X_train_features, X_test_features = make_tfidf_pos_matrices(
            X_train_text,
            X_test_text
        )

    elif representation == "bert":
        X_train_features = bert_train
        X_test_features = bert_test

    else:
        raise ValueError("Unknown representation.")

   
    if representation == "bert":
        model = LogisticRegression(
            C=C_value,
            max_iter=1000,
            solver="lbfgs",
            multi_class="multinomial"
        )
    else:
        model = LogisticRegression(
            C=C_value,
            max_iter=500,
            solver="liblinear"
        )

    model.fit(X_train_features, y_train)
    y_pred = model.predict(X_test_features)

    return y_pred
def tune_C_with_inner_cv(
    X_train_text,
    y_train,
    representation,
    C_values,
    inner_folds_number,
    random_state,
    bert_train_all=None
):
    inner_folds = make_stratified_folds(
        y_train,
        inner_folds_number,
        random_state
    )

    all_indices = np.arange(len(y_train))

    best_C = None
    best_score = -1

    for C_value in C_values:
        fold_scores = []

        for valid_indices in inner_folds:
            train_indices = np.setdiff1d(all_indices, valid_indices)

            X_inner_train_text = X_train_text[train_indices]
            y_inner_train = y_train[train_indices]

            X_inner_valid_text = X_train_text[valid_indices]
            y_inner_valid = y_train[valid_indices]

            if representation == "bert":
                bert_inner_train = bert_train_all[train_indices]
                bert_inner_valid = bert_train_all[valid_indices]
            else:
                bert_inner_train = None
                bert_inner_valid = None

            y_valid_pred = train_and_predict(
                X_train_text=X_inner_train_text,
                y_train=y_inner_train,
                X_test_text=X_inner_valid_text,
                representation=representation,
                C_value=C_value,
                bert_train=bert_inner_train,
                bert_test=bert_inner_valid
            )

            macro_f1 = calculate_macro_f1(y_inner_valid, y_valid_pred)
            fold_scores.append(macro_f1)

        average_score = np.mean(fold_scores)

        print("C =", C_value, "| inner macro-F1 =", round(average_score, 4))

        if average_score > best_score:
            best_score = average_score
            best_C = C_value

    return best_C, best_score
def run_nested_cv_one_setting(data, context_column, representation, C_values, bert_embeddings=None):
    X = data[context_column].astype(str).to_numpy()
    y = data["label"].to_numpy()

    outer_folds = make_stratified_folds(
        y,
        OUTER_FOLDS,
        random_state=RANDOM_STATE
    )

    all_indices = np.arange(len(y))

    fold_rows = []
    per_class_rows = []

    for outer_fold_number, test_indices in enumerate(outer_folds, start=1):
        print()
        print("====================================")
        print("Context:", context_column)
        print("Representation:", representation)
        print("Outer fold:", outer_fold_number)
        print("====================================")

        train_indices = np.setdiff1d(all_indices, test_indices)

        X_train_text = X[train_indices]
        y_train = y[train_indices]

        X_test_text = X[test_indices]
        y_test = y[test_indices]

        if representation == "bert":
            bert_train_all = bert_embeddings[train_indices]
            bert_test = bert_embeddings[test_indices]
        else:
            bert_train_all = None
            bert_test = None

        best_C, best_inner_macro_f1 = tune_C_with_inner_cv(
            X_train_text=X_train_text,
            y_train=y_train,
            representation=representation,
            C_values=C_values,
            inner_folds_number=INNER_FOLDS,
            random_state=outer_fold_number,
            bert_train_all=bert_train_all
        )

        if representation == "bert":
            bert_train = bert_embeddings[train_indices]
            bert_test = bert_embeddings[test_indices]
        else:
            bert_train = None
            bert_test = None

        y_test_pred = train_and_predict(
            X_train_text=X_train_text,
            y_train=y_train,
            X_test_text=X_test_text,
            representation=representation,
            C_value=best_C,
            bert_train=bert_train,
            bert_test=bert_test
        )

        test_accuracy = calculate_accuracy(y_test, y_test_pred)
        test_macro_f1 = calculate_macro_f1(y_test, y_test_pred)
        test_weighted_f1 = calculate_weighted_f1(y_test, y_test_pred)

        print("Best C:", best_C)
        print("Test accuracy:", test_accuracy)
        print("Test macro-F1:", test_macro_f1)
        print("Test weighted-F1:", test_weighted_f1)

        fold_row = {
            "context_level": context_column,
            "representation": representation,
            "outer_fold": outer_fold_number,
            "best_C": best_C,
            "inner_macro_f1": best_inner_macro_f1,
            "test_accuracy": test_accuracy,
            "test_macro_f1": test_macro_f1,
            "test_weighted_f1": test_weighted_f1
        }

        fold_rows.append(fold_row)

        per_class_f1 = calculate_per_class_f1(y_test, y_test_pred)

        for label in per_class_f1:
            per_class_rows.append({
                "context_level": context_column,
                "representation": representation,
                "outer_fold": outer_fold_number,
                "label": label,
                "f1": per_class_f1[label]
            })

    fold_results = pd.DataFrame(fold_rows)
    per_class_results = pd.DataFrame(per_class_rows)

    return fold_results, per_class_results

Run Representation Experiments

In [8]:
representations = ["bert"]

all_fold_results = []
all_per_class_results = []

bert_embedding_storage = {}


if USE_BERT:
    for context_column in CONTEXT_COLUMNS:
        print("=" * 50)
        print("Creating BERT embeddings for:", context_column)
        print("=" * 50)

        text_list = data[context_column].astype(str).tolist()

        bert_embeddings = make_bert_embeddings(
            text_list=text_list,
            model_name=BERT_MODEL,
            batch_size=BERT_BATCH_SIZE,
            max_length=BERT_MAX_LENGTH
        )

        print(f"  -> shape: {bert_embeddings.shape}, dtype: {bert_embeddings.dtype}")
        bert_embedding_storage[context_column] = bert_embeddings
for context_column in CONTEXT_COLUMNS:
    for representation in representations:
        if representation == "bert":
            bert_embeddings = bert_embedding_storage[context_column]
        else:
            bert_embeddings = None

        fold_results, per_class_results = run_nested_cv_one_setting(
            data=data,
            context_column=context_column,
            representation=representation,
            C_values=C_VALUES,
            bert_embeddings=bert_embeddings
        )

        all_fold_results.append(fold_results)
        all_per_class_results.append(per_class_results)

Creating BERT embeddings for: L1
BERT device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

C = 1 | inner macro-F1 = 0.7295


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


C = 10 | inner macro-F1 = 0.6916


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best C: 0.1
Test accuracy: 0.7565
Test macro-F1: 0.7563579162891394
Test weighted-F1: 0.7563579162891392


result

In [9]:
final_fold_results = pd.concat(all_fold_results, axis=0)
final_per_class_results = pd.concat(all_per_class_results, axis=0)


def make_summary(fold_results):
    summary_rows = []

    context_levels = sorted(fold_results["context_level"].unique())
    representations = sorted(fold_results["representation"].unique())

    for context in context_levels:
        for representation in representations:
            subset = fold_results[
                (fold_results["context_level"] == context) &
                (fold_results["representation"] == representation)
            ]

            row = {
                "context_level": context,
                "representation": representation,
                "accuracy_mean": subset["test_accuracy"].mean(),
                "accuracy_std": subset["test_accuracy"].std(),
                "macro_f1_mean": subset["test_macro_f1"].mean(),
                "macro_f1_std": subset["test_macro_f1"].std(),
                "weighted_f1_mean": subset["test_weighted_f1"].mean(),
                "weighted_f1_std": subset["test_weighted_f1"].std()
            }

            summary_rows.append(row)

    return pd.DataFrame(summary_rows)


summary_results = make_summary(final_fold_results)


SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/SML"

final_fold_results.to_csv(f"{SAVE_DIR}/logistic_bert_fold_results.csv", index=False)
summary_results.to_csv(f"{SAVE_DIR}/logistic_bert_summary_results.csv", index=False)
final_per_class_results.to_csv(f"{SAVE_DIR}/logistic_bert_per_class_f1_results.csv", index=False)

print(f"Saved logistic_bert_fold_results.csv     to {SAVE_DIR}")
print(f"Saved logistic_bert_summary_results.csv  to {SAVE_DIR}")
print(f"Saved logistic_bert_per_class_f1_results.csv to {SAVE_DIR}")

summary_results

Saved logistic_bert_fold_results.csv     to /content/drive/MyDrive/Colab Notebooks/SML
Saved logistic_bert_summary_results.csv  to /content/drive/MyDrive/Colab Notebooks/SML
Saved logistic_bert_per_class_f1_results.csv to /content/drive/MyDrive/Colab Notebooks/SML


,context_level,representation,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,L1,bert,0.57105,0.010245,0.570379,0.010031,0.570379,0.010031
1,L2,bert,0.70340,0.009207,0.702335,0.009429,0.702335,0.009429
2,L3,bert,0.73345,0.008335,0.732256,0.009037,0.732256,0.009037
3,L4,bert,0.75180,0.010001,0.750438,0.010427,0.750438,0.010427
